# Bipolarity metrics pilot: 4 approaches compared on 20 articles

Tests four approaches to detecting how much a news article frames an issue as strictly two opposing/mutually-exclusive sides (see `src/bipolarity.py` and `src/llm_bipolarity.py` for full docstrings):

**CPU-only, no LLM needed (sections 1-5):**
1. **`dichotomy_marker_density`** -- keyword/pattern density of explicit binary-framing language (`either...or`, `vs`, `two sides`, ...).
2. **`entity_sentiment_gap`** -- VADER sentiment gap between the two most-mentioned named entities, a proxy for implicit bipolar framing.
3. **`antonym_cooccurrence_density`** -- density of WordNet antonym pairs both appearing in the article.

**Requires the local LLM / a GPU runtime (section 6):**
4. **`llm_dichotomy_density`** -- Llama-3.1-8B-Instruct is asked to find every instance of **"erasure of complexities"**: the collapse of plural, multidimensional political/social identity into a single, seemingly natural and inescapable two-camp antagonism (not just "two options on an issue" -- a group's whole identity being reduced to one axis of opposition, or a stark partisan-division statistic being cited as if it proves total, natural opposition without the nuancing context that would complicate it). This is a distinct, more theoretically specific definition than the SemEval `black_and_white` fallacy used in the main propaganda pipeline -- see the docstring in `src/llm_bipolarity.py` for the full theoretical grounding and guardrails. One LLM call per article -- the only one of the four that costs GPU time / Colab compute credits.

None of the four has been validated against human-labeled bipolarity judgments -- treat them as proxies to compare, not ground truth.

**If you're out of Colab GPU credits, skip section 6** -- sections 1-5 run the three CPU-only metrics and are fully independent of it.

## 1. Install dependencies (CPU-only metrics)

In [ ]:
!pip install -q spacy nltk pandas
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("vader_lexicon", quiet=True)

## 2. Clone this repo (branch: `claude/bipolarity-metrics`)

In [ ]:
import os

REPO_URL = "https://github.com/hrauxloh/DAAD_Destructive_polarization"
BRANCH = "claude/bipolarity-metrics"
REPO_DIR = "/content/DAAD_Destructive_polarization"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

## 3. Load the first 20 articles

In [ ]:
import csv
import pandas as pd

with open("australia_498sample_climatechange.csv", newline="", encoding="utf-8") as f:
    articles = list(csv.DictReader(f))[:20]

print(f"loaded {len(articles)} articles")

## 4. Run the three CPU-only metrics

In [ ]:
from src.bipolarity import compute_bipolarity_table

cpu_rows = compute_bipolarity_table(articles)
cpu_df = pd.DataFrame(cpu_rows)
cpu_df

## 5. Compare the three CPU-only approaches
Correlation between the three main scores -- do they agree on which articles are "most bipolar," or are they picking up on different things?

In [ ]:
cpu_cols = ["dichotomy_marker_density", "entity_sentiment_gap", "antonym_cooccurrence_density"]
print(cpu_df[cpu_cols].corr())
cpu_df[["document_id", "title"] + cpu_cols]

## 6. LLM-based "erasure of complexities" detection (needs GPU / uses Colab credits)
**Skip this section if you don't have GPU runtime available right now** -- sections 1-5 above already give you 3 of the 4 comparison points without it.

This installs and loads Llama-3.1-8B-Instruct the same way as `notebooks/colab_propaganda_poc.ipynb`. Set `Runtime > Change runtime type > T4 GPU` before running this section.

In [ ]:
import subprocess

def gpu_available():
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True)
        return True
    except Exception:
        return False

USE_GPU = gpu_available()
print(f"GPU detected: {USE_GPU}")

!pip install -q huggingface_hub

# --force-reinstall --no-cache-dir matters here: if llama-cpp-python is
# already installed from an earlier cell run / earlier session on this same
# Colab VM (e.g. a CUDA build from before a runtime switch), a plain
# `pip install` sees the requirement as already satisfied and silently
# skips reinstalling -- leaving the WRONG build in place even though this
# cell "ran successfully". Forcing a clean reinstall avoids that.
if USE_GPU:
    !pip install -q --force-reinstall --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
    # Fallback if the prebuilt wheel doesn't match Colab's current CUDA version (slower, ~5-10 min):
    # !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir llama-cpp-python
else:
    print("No GPU detected -- installing CPU-only llama-cpp-python. "
          "Generation will be noticeably slower than on a T4 GPU.")
    !pip install -q --force-reinstall --no-cache-dir llama-cpp-python

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
MODEL_FILE = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(model_path)

In [ ]:
import subprocess

def gpu_memory_used_mb():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        return int(out.splitlines()[0])
    except Exception:
        return None

def load_llama(n_gpu_layers):
    from llama_cpp import Llama
    return Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=8192,
        n_threads=os.cpu_count(),
        verbose=False,
    )

mem_before = gpu_memory_used_mb() if USE_GPU else None

try:
    llm = load_llama(-1 if USE_GPU else 0)
except OSError as e:
    if "libcudart" not in str(e) and "libcuda" not in str(e):
        raise
    # Self-heal: the installed build is CUDA-linked but this runtime has no
    # CUDA runtime to satisfy it (e.g. pip silently kept a stale CUDA build
    # from an earlier session). Force a genuinely CPU-only reinstall and
    # retry once.
    print(f"CUDA library load failed ({e}). Forcing a clean CPU-only "
          "reinstall of llama-cpp-python and retrying...")
    !pip install -q --force-reinstall --no-cache-dir llama-cpp-python
    USE_GPU = False
    mem_before = None
    llm = load_llama(0)

if not USE_GPU:
    print("Running on CPU -- generation will be noticeably slower than on a T4.")
else:
    mem_after = gpu_memory_used_mb()
    if mem_before is not None and mem_after is not None:
        print(f"GPU memory delta: {mem_after - mem_before} MB")
        if mem_after - mem_before < 500:
            print("WARNING: model does not appear to be on the GPU (see notebooks/colab_propaganda_poc.ipynb troubleshooting notes)")

In [ ]:
from src.llm_bipolarity import compute_llm_dichotomy_table

def generate_fn(messages, max_tokens=1024, temperature=0.0):
    resp = llm.create_chat_completion(messages=messages, max_tokens=max_tokens, temperature=temperature)
    return resp["choices"][0]["message"]["content"]

llm_rows = compute_llm_dichotomy_table(articles, generate_fn)
llm_df = pd.DataFrame(llm_rows)
llm_df[["document_id", "title", "llm_dichotomy_instance_count", "llm_dichotomy_density", "llm_dichotomy_parse_error"]]

## 7. Compare all four approaches

In [ ]:
all_df = cpu_df.merge(llm_df.drop(columns=["title"]), on="document_id")
all_cols = cpu_cols + ["llm_dichotomy_density"]

print("=== Correlation across all 4 approaches ===")
print(all_df[all_cols].corr())

for col in all_cols:
    print(f"\n=== Top 5 by {col} ===")
    for _, row in all_df.nlargest(5, col).iterrows():
        print(f"  [{row['document_id']}] {row[col]:.4f} -- {row['title'][:70]}")

all_df.to_csv("aus_bipolarity_pilot_4way.csv", index=False)
all_df

## 8. Download the results

In [ ]:
from google.colab import files
files.download("aus_bipolarity_pilot_4way.csv")

## Notes / limitations
- **`dichotomy_marker_density` was 0 for all 20 articles** in initial local testing -- explicit binary phrasing is rare in this climate-news corpus. Informative limitation, not a bug.
- `entity_sentiment_gap` requires at least 2 distinct named entities; articles with fewer show `None`.
- `antonym_cooccurrence_density` uses WordNet antonym relations without word-sense disambiguation, so it can surface an occasional spurious pair -- check `antonym_pairs_sample` before trusting a high score.
- `llm_dichotomy_density` uses a theoretically-grounded "erasure of complexities" definition (see `src/llm_bipolarity.py` docstring) -- a DIFFERENT, more specific construct than the SemEval `black_and_white` fallacy used in the main propaganda pipeline (`notebooks/colab_propaganda_poc.ipynb`). Don't expect these two to agree closely; they're intentionally measuring different things.
- None of these four has been validated against human bipolarity judgments -- treat all of them as exploratory signals to compare, not ground truth.
- Section 6 costs GPU time/Colab compute credits; sections 1-5 do not.